# Pérdida esperada vs dependencia extrema

**Laboratorio crítico | Datos sintéticos | No evidencia empírica**

Dos portafolios con iguales PD, LGD y EAD pueden exhibir probabilidades muy diferentes de pérdida conjunta. El riesgo de cola no queda determinado por EL.

## 1. Hipótesis rivales y supuestos
**Base:** defaults independientes. **Alternativa:** defaults generados por un factor normal común. Para cada obligación, PD=0.1, LGD=1, EAD=100. El modelo no incluye contagio dinámico ni LGD estocástica.

## 2. Derivación
Por linealidad de esperanza, E[L]=Σ E[L_i]=20 sin exigir independencia. En cambio, P(D1∩D2)=0.01 bajo independencia y depende de la cópula en el escenario correlacionado. Se simula Z_i=√ρ F+√(1−ρ) ε_i.


In [ ]:
# Preparación reproducible y funciones del experimento
import random, math
from statistics import NormalDist
import matplotlib.pyplot as plt
threshold=NormalDist().inv_cdf(.1)
def simulate(rho,n=25000,seed=12):
    rng=random.Random(seed)
    losses=[];joint=0
    for _ in range(n):
        f=rng.gauss(0,1)
        ds=[math.sqrt(rho)*f+math.sqrt(1-rho)*rng.gauss(0,1)<threshold for _ in range(2)]
        joint+=int(all(ds));losses.append(100*sum(ds))
    return sum(losses)/n,joint/n



## 3. Experimento: escenario base y ruptura de supuestos

La comparación mantiene constantes los demás parámetros. Un cambio de resultado al violar un supuesto **no** invalida una proposición que lo requiere.


In [ ]:
rhos=[0,.2,.4,.6,.8,.95]
results=[simulate(r) for r in rhos]
assert all(abs(mean-20)<1.2 for mean,joint in results)
assert results[-1][1]>results[0][1]+.03
plt.figure(figsize=(9,4.5))
plt.plot(rhos,[p for m,p in results],'o-',label='P(default conjunto)')
plt.axhline(.01,color='gray',linestyle='--',label='Independencia')
plt.xlabel('Correlación latente ρ');plt.ylabel('Probabilidad conjunta')
plt.title('Misma EL, diferente dependencia de cola')
plt.legend();plt.grid(alpha=.2);plt.show()
print('rho, mean loss, joint default',list(zip(rhos,results)))


## 4. Interpretación y falsación

El escenario con factor común genera mayor concentración de pérdidas. Una PD marginal no revela la dependencia, y una identidad de EL no es una distribución de pérdidas. Contrastar con t-cópulas, redes de contagio y LGD cíclica; exigir pruebas de estrés y análisis de sensibilidad antes de aplicar a capital económico.

## 5. Ejercicios de investigación

1. Derive el estimando sin recurrir al código. ¿Qué hipótesis hace falta?
2. Modifique un solo parámetro y describa una predicción que distinga mecanismos.
3. Diseñe un control negativo o placebo; explique qué resultado invalidaría su interpretación.
4. Indique una fuente de datos y el tratamiento de vintages, sesgo de selección y variables omitidas.
5. Explique por qué una simulación reproducible no constituye evidencia empírica.

## 6. Bibliografía de referencia
- Vasicek (2002), *Loan Portfolio Value*.
- McNeil, Frey & Embrechts (2015), *Quantitative Risk Management*.

**Límites:** No se estiman parámetros con datos observados. Los intervalos de incertidumbre, si se usan, deben justificarse por el DGP y no por la apariencia de las figuras.
